# 🎙️ Notebook 1 — Transcribe Comorian Audio
### Upload Audio → Transcribe with Whisper Large-v3 → Save to Drive

---
## 📋 Steps
| Step | What happens |
|------|--------------|
| 1 | Check GPU & install packages |
| 2 | Mount Google Drive |
| 3 | Load faster-whisper large-v3 |
| 4 | Load audio from Drive |
| 5 | Transcribe with Whisper |
| 6 | Save transcriptions to Drive |

> 💡 **Before starting:** Go to `Runtime` → `Change runtime type` → Select `T4 GPU` → Save

---
### 🔬 Why faster-whisper large-v3?
| Feature | Why it matters for Comorian |
|---------|---------------------------|
| 99+ languages | Swahili is well-represented → Comorian benefits (same Bantu/Sabaki family) |
| French support | Handles French loanwords naturally (ordinateuri, aplikasyon, etc.) |
| 4x faster | CTranslate2 backend → fits T4 GPU on free Colab |
| ~6 GB VRAM | Runs comfortably on T4 (16GB) or Kaggle P100 |

## ✅ Step 1: Check GPU & Install Packages

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
    print(f'✅ GPU ready: {gpu_name} ({gpu_mem} GB)')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

print(f'\n🔧 Device: {device}')

In [ ]:
print('📦 Installing packages...')
!pip install faster-whisper pydub -q
!apt-get install -y ffmpeg -q 2>/dev/null
print('✅ All packages installed!')

## ✅ Step 2: Mount Google Drive

Mounts your Drive once per session — no more permission popups on every run.

Your dataset folder structure:
```
My Drive/
  └── SHIKOMORI_dataset/
      ├── raw/              ← your audio recordings
      └── output/           ← transcriptions go here
```

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────
# 📁 SET YOUR FOLDER PATH HERE
# Change this if your folder is named differently
# ─────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/SHIKOMORI_dataset'

DRIVE_RAW    = f'{DRIVE_ROOT}/raw'
DRIVE_OUTPUT = f'{DRIVE_ROOT}/output'

for d in [DRIVE_RAW, DRIVE_OUTPUT]:
    os.makedirs(d, exist_ok=True)

print(f'\n✅ Drive mounted!')
print(f'   📂 {DRIVE_ROOT}')
print(f'   ├── raw/    ✅')
print(f'   └── output/ ✅')

## ✅ Step 3: Load faster-whisper Large-v3

- Uses **float16** on GPU for speed + low VRAM (~6 GB)
- 1.55 billion parameters, 99+ languages
- Best zero-shot multilingual ASR model available

In [ ]:
from faster_whisper import WhisperModel
import time

MODEL_SIZE   = 'large-v3'
COMPUTE_TYPE = 'float16'

if device == 'cpu':
    COMPUTE_TYPE = 'int8'
    print('⚠️  Running on CPU — using int8 quantization (slower but works)')

print(f'🔄 Loading faster-whisper {MODEL_SIZE} ({COMPUTE_TYPE})...')
print(f'   This may take 2-3 minutes on first run (downloading ~3 GB)...')

whisper_model = WhisperModel(
    MODEL_SIZE,
    device=device,
    compute_type=COMPUTE_TYPE
)

print(f'\n✅ faster-whisper {MODEL_SIZE} loaded on {device.upper()}!')
print(f'   Compute type  : {COMPUTE_TYPE}')
print(f'   Parameters    : ~1.55 billion')
print(f'   Languages     : 99+')
print(f'   Engine        : CTranslate2 (4x faster than original Whisper)')

## ✅ Step 4: Load Audio From Drive

Put your audio files in the Drive `raw/` folder, then run the next cell.

### Load from Drive

Put your audio files in `My Drive/SHIKOMORI_dataset/raw/` and run this cell.

In [ ]:
SUPPORTED = ('.mp3', '.wav', '.m4a', '.ogg', '.flac', '.mp4', '.webm')

audio_files = [
    f for f in os.listdir(DRIVE_RAW)
    if f.lower().endswith(SUPPORTED) and not f.startswith('.')
]

if not audio_files:
    print('⚠️  No audio files found in Drive raw/ folder')
    print(f'   Upload files to: {DRIVE_RAW}')
else:
    print(f'✅ Found {len(audio_files)} audio file(s) in Drive:\n')
    for f in sorted(audio_files):
        size_kb = os.path.getsize(f'{DRIVE_RAW}/{f}') / 1024
        print(f'   • {f}  ({size_kb:.0f} KB)')

## ✅ Step 5: Transcribe All Audio Files

Transcribes every audio file in `raw/` directly — no slowdown or splitting needed for short recordings (1-50s).
Each file is tried twice: auto-detect first, then forced Swahili as a fallback for mixed Comorian speech.

### 🔬 Whisper settings for Comorian
| Setting | Value | Why |
|---|---|---|
| `language` | **auto → sw fallback** | Auto-detect first, then retry with Swahili if needed |
| `vad_filter` | **True** | Skips silence automatically |
| `repetition_penalty` | **1.2** | Prevents looping repeated words |

In [ ]:
# ─────────────────────────────────────────────────────────
# 🔍 FIND AUDIO FILES
# ─────────────────────────────────────────────────────────
SUPPORTED = ('.mp3', '.wav', '.m4a', '.ogg', '.flac', '.mp4', '.webm')
audio_files = sorted([
    f for f in os.listdir(DRIVE_RAW)
    if f.lower().endswith(SUPPORTED) and not f.startswith('.')
])

if not audio_files:
    raise FileNotFoundError(
        f'❌ No audio files in {DRIVE_RAW}\n'
        '   Upload audio files to the Drive raw/ folder first'
    )

TRANSCRIBE_CONFIG = dict(
    task='transcribe',
    beam_size=5,
    best_of=5,
    no_speech_threshold=0.3,
    log_prob_threshold=-1.5,
    compression_ratio_threshold=3.0,
    condition_on_previous_text=True,
    word_timestamps=True,
    repetition_penalty=1.2,
    vad_filter=True,
    vad_parameters=dict(
        min_silence_duration_ms=500,
        speech_pad_ms=200,
    ),
    multilingual=True,
)

PASS_CONFIGS = [
    {'name': 'auto-detect', 'language': None},
    {'name': 'forced-sw', 'language': 'sw'},
]

def run_transcription_pass(audio_path, pass_name, language):
    kwargs = dict(TRANSCRIBE_CONFIG)
    kwargs['language'] = language

    try:
        try:
            segments, info = whisper_model.transcribe(audio_path, **kwargs)
        except TypeError:
            kwargs.pop('multilingual', None)
            segments, info = whisper_model.transcribe(audio_path, **kwargs)

        segments = list(segments)
        full_text = []
        timestamps_text = []
        avg_probs = []

        for seg in segments:
            seg_text = seg.text.strip()
            if not seg_text:
                continue
            full_text.append(seg_text)
            avg_prob = seg.avg_logprob
            avg_probs.append(avg_prob)
            icon = '🟢' if avg_prob > -0.3 else ('🟡' if avg_prob > -0.7 else '🔴')
            timestamps_text.append(
                f'[{seg.start:.1f}s → {seg.end:.1f}s]  '
                f'{icon} ({avg_prob:.2f})  {seg_text}'
            )

        text = ' '.join(full_text) if full_text else '[no speech detected]'
        avg_logprob = sum(avg_probs) / len(avg_probs) if avg_probs else float('-inf')

        return {
            'pass_name': pass_name,
            'requested_language': language or 'auto',
            'detected_language': getattr(info, 'language', 'unknown'),
            'language_probability': getattr(info, 'language_probability', None),
            'segment_count': len(full_text),
            'avg_logprob': avg_logprob,
            'text': text,
            'timestamps_text': timestamps_text,
            'error': None,
            'has_speech': bool(full_text),
        }
    except Exception as e:
        return {
            'pass_name': pass_name,
            'requested_language': language or 'auto',
            'detected_language': 'unknown',
            'language_probability': None,
            'segment_count': 0,
            'avg_logprob': float('-inf'),
            'text': f'[error: {e}]',
            'timestamps_text': [],
            'error': str(e),
            'has_speech': False,
        }

def choose_best_result(results):
    usable = [r for r in results if r['error'] is None and r['has_speech']]
    if usable:
        return max(
            usable,
            key=lambda r: (
                r['segment_count'],
                r['avg_logprob'],
                1 if r['pass_name'] == 'auto-detect' else 0,
            ),
        )

    no_error = [r for r in results if r['error'] is None]
    if no_error:
        return max(
            no_error,
            key=lambda r: (
                r['segment_count'],
                r['avg_logprob'],
                1 if r['pass_name'] == 'auto-detect' else 0,
            ),
        )

    return results[0]

print(f'\n🚀 Transcribing {len(audio_files)} file(s)...')
print(f'   Model : faster-whisper {MODEL_SIZE}')
print('   Pass 1: auto-detect')
print('   Pass 2: forced Swahili (sw) fallback')
print('=' * 55)

summary = []

for idx, fname in enumerate(audio_files, 1):
    fpath = f'{DRIVE_RAW}/{fname}'
    stem  = Path(fname).stem

    print(f'\n[{idx}/{len(audio_files)}] 🎙️  {fname}')
    t0 = time.time()

    results = [run_transcription_pass(fpath, cfg['name'], cfg['language']) for cfg in PASS_CONFIGS]
    chosen = choose_best_result(results)

    elapsed = round(time.time() - t0, 1)
    words = len(chosen['text'].split())
    lang_prob = chosen['language_probability']
    lang_prob_text = f'{lang_prob:.2f}' if isinstance(lang_prob, float) else 'n/a'

    # Save plain text
    txt_path = f'{DRIVE_OUTPUT}/{stem}_transcript.txt'
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(f'Source File   : {fname}\n')
        f.write(f'Model         : faster-whisper {MODEL_SIZE}\n')
        f.write(f'Selected Pass : {chosen["pass_name"]}\n')
        f.write(f'Requested Lang: {chosen["requested_language"]}\n')
        f.write(f'Detected Lang : {chosen["detected_language"]}\n')
        f.write(f'Lang Prob     : {lang_prob_text}\n')
        f.write(f'Avg Logprob   : {chosen["avg_logprob"]:.2f}\n')
        f.write(f'Transcribed   : {time.strftime("%Y-%m-%d %H:%M")}\n')
        f.write('=' * 55 + '\n\n')
        f.write(chosen['text'])

    # Save timestamped version
    ts_path = f'{DRIVE_OUTPUT}/{stem}_timestamps.txt'
    with open(ts_path, 'w', encoding='utf-8') as f:
        f.write(f'Source        : {fname}\n')
        f.write(f'Selected Pass : {chosen["pass_name"]}\n')
        f.write(f'Detected Lang : {chosen["detected_language"]}\n')
        f.write(f'Avg Logprob   : {chosen["avg_logprob"]:.2f}\n')
        f.write('🟢 > -0.3 confident | 🟡 uncertain | 🔴 needs correction\n')
        f.write('=' * 55 + '\n\n')
        f.write('\n'.join(chosen['timestamps_text']))

    summary.append({
        'file': fname,
        'words': words,
        'pass_name': chosen['pass_name'],
        'detected_language': chosen['detected_language'],
    })
    print(f'   ✅ {elapsed}s | {words} words | {chosen["pass_name"]}')
    print(f'   🌍 detected: {chosen["detected_language"]} (p={lang_prob_text})')
    print(f'   📄 {chosen["text"][:100]}...' if len(chosen['text']) > 100 else f'   📄 {chosen["text"]}')

print('\n' + '=' * 55)
print('🎉 ALL TRANSCRIPTIONS COMPLETE!')
print(f'   Saved to: {DRIVE_OUTPUT}/\n')
for s in summary:
    print(f"   • {s['file']} → {s['words']} words | {s['pass_name']} | {s['detected_language']}")

## ✅ Step 6: Check Your Results

Your transcriptions are already saved in Drive — no extra upload step needed!

In [ ]:
print('📂 Transcription files in Drive:\n')
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    if f.endswith('.txt'):
        size_kb = os.path.getsize(f'{DRIVE_OUTPUT}/{f}') / 1024
        print(f'   📄 {f}  ({size_kb:.1f} KB)')

print(f'\n📁 Location: {DRIVE_OUTPUT}')
print('\n💡 You can also browse them in the Files panel on the left →')

---
## ✅ Done! Next Steps

1. **Review** the `_timestamps.txt` files — correct 🟡 and 🔴 segments
2. **Upload more audio** — add new files to Drive `raw/` and run the notebook again
3. When you have **10-50 hours corrected** → open **`02_finetune_whisper.ipynb`**

---
*Built for the Comorian (Shikomori) Language Preservation Project* 🇰🇲